# PointDiT in motion: Kaggle runner

Needs a GPU accelerator, Internet on, the "PointDiT TUM RGB-D Subset" dataset attached, a Kaggle secret `GITHUB_TOKEN` while the repo is private (fine-grained, read-only Contents on this repo), and a Kaggle secret `HF_TOKEN` for an account with access to `facebook/dinov3-vitl16-pretrain-lvd1689m`.

In [ ]:
PROJECT_REPO = "https://github.com/Sainava/pointdit-in-motion.git"
DINOV3_COMMIT = "6876159a11b4df116f30f667f8c9888617df0751"
OUTPUT_DIR = "/kaggle/working/outputs"

PROJECT_DIR = "/kaggle/working/pointdit-in-motion"
FORK_DIR = f"{PROJECT_DIR}/third_party/pointdit"
DINOV3_DIR = f"{FORK_DIR}/third_party/dinov3"

In [ ]:
import base64
import os

from kaggle_secrets import UserSecretsClient


def github_token() -> str | None:
    try:
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None


# Only needed while the repo is private. Git reads the token from the environment, so it
# never appears in a command, in cell output, or in the clone's .git/config.
token = github_token()
if token:
    basic_auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    os.environ["GIT_CONFIG_COUNT"] = "1"
    os.environ["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
    os.environ["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {basic_auth}"
    del basic_auth
print("GitHub auth:", "token" if token else "none (public repo)")
del token

# Check for .git, not just the directory: after a failed clone, the DINOv3 clone below
# creates PROJECT_DIR as a parent, which would otherwise skip this clone on every rerun.
if not os.path.isdir(f"{PROJECT_DIR}/.git"):
    !rm -rf {PROJECT_DIR}
    !git clone --recurse-submodules {PROJECT_REPO} {PROJECT_DIR}
if not os.path.isdir(f"{PROJECT_DIR}/.git"):
    raise RuntimeError(f"Cloning {PROJECT_REPO} failed; see the git output above.")
!git -C {PROJECT_DIR} submodule update --init --recursive
%cd {PROJECT_DIR}
!echo "project $(git -C {PROJECT_DIR} rev-parse HEAD)"
!echo "fork    $(git -C {FORK_DIR} rev-parse HEAD)"

In [ ]:
if not os.path.isdir(DINOV3_DIR):
    !git clone -q https://github.com/facebookresearch/dinov3.git {DINOV3_DIR}
if DINOV3_COMMIT:
    !git -C {DINOV3_DIR} checkout -q {DINOV3_COMMIT}
else:
    print("DINOV3_COMMIT is empty; paste the hash below into the config cell to pin it.")
!echo "dinov3  $(git -C {DINOV3_DIR} rev-parse HEAD)"

In [ ]:
!pip install -q -r {FORK_DIR}/requirements.txt -r {PROJECT_DIR}/requirements.txt

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!python {PROJECT_DIR}/scripts/setup_weights.py

In [ ]:
import sys

sys.path.insert(0, f"{PROJECT_DIR}/src")
from pim.data.tum import find_sequences

sequences = find_sequences("/kaggle/input")
for sequence in sequences:
    print(sequence)

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python {PROJECT_DIR}/scripts/smoke_test.py --sequence "{sequences[0]}" --output-dir {OUTPUT_DIR}/smoke

In [ ]:
import torch

print("torch  ", torch.__version__)
print("CUDA   ", torch.version.cuda)
print("GPU    ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
!echo "project $(git -C {PROJECT_DIR} rev-parse HEAD)"
!echo "fork    $(git -C {FORK_DIR} rev-parse HEAD)"
!echo "dinov3  $(git -C {DINOV3_DIR} rev-parse HEAD)"